# Playing with the VLM chatbot API

This notebook drives the API directly — no UI, no Docker. Run the cells in order.

**Before you start**, make sure the API is running on `http://localhost:8000`:

```bash
cd PP8_API_Docker/api
uv sync
uv run uvicorn main:app --reload --port 8000
```

## 1. Setup and health check

In [ ]:
import base64
import io

import requests
from PIL import Image

API_URL = "http://localhost:8000"

# Sanity check — the API should respond with {"status": "ok"}
r = requests.get(f"{API_URL}/health", timeout=5)
r.raise_for_status()
print(r.json())

## 2. A single text request

Build a one-message conversation, send it, parse the response.

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "In one sentence, what is a vision-language model?"}
    ],
    "max_new_tokens": 80,
    "temperature": 0.3,
}

r = requests.post(f"{API_URL}/chat", json=payload, timeout=180)
r.raise_for_status()
data = r.json()

print("reply:", data["reply"])
print("latency:", data["generation_time_ms"], "ms")

## 3. Multi-turn: keeping history on the client

The API is **stateless** — it remembers nothing between requests. The conversation lives here, in this notebook, as a Python list. On every turn we:

1. append the new user message,
2. send the **whole list** to the API,
3. append the assistant's reply.

This is the same pattern the Gradio UI uses internally.

In [ ]:
def chat(messages, image_base64=None, **kwargs):
    """Send a conversation to the API and return the assistant's reply."""
    payload = {"messages": messages, **kwargs}
    if image_base64 is not None:
        payload["image_base64"] = image_base64
    r = requests.post(f"{API_URL}/chat", json=payload, timeout=180)
    r.raise_for_status()
    return r.json()["reply"]


history = []

# Turn 1
history.append({"role": "user", "content": "My name is Ada. Remember it."})
reply = chat(history, max_new_tokens=60)
history.append({"role": "assistant", "content": reply})
print("Assistant:", reply)

In [ ]:
# Turn 2 — the API has no memory, but the history we send does
history.append({"role": "user", "content": "What is my name?"})
reply = chat(history, max_new_tokens=40)
history.append({"role": "assistant", "content": reply})
print("Assistant:", reply)

print("\nFull history so far:")
for m in history:
    print(f"  {m['role']}: {m['content']}")

Try commenting out the `history.append` for the assistant reply and re-running — you'll see the model lose track of the conversation. That's the whole point: history is **our** responsibility.

## 4. Sending an image

The image travels as a **base64-encoded string** inside the JSON body. We build a tiny synthetic image here, but you can replace it with any PIL image.

In [ ]:
from PIL import ImageDraw

# Make a 224x224 image with a red square on a white background
img = Image.new("RGB", (224, 224), "white")
draw = ImageDraw.Draw(img)
draw.rectangle([60, 60, 164, 164], fill="red")
img

In [ ]:
def encode_image(image: Image.Image) -> str:
    buf = io.BytesIO()
    image.convert("RGB").save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")


img_b64 = encode_image(img)
print("base64 size:", len(img_b64), "chars")

reply = chat(
    [{"role": "user", "content": "What do you see in this image? Be brief."}],
    image_base64=img_b64,
    max_new_tokens=80,
)
print("Assistant:", reply)

## 5. Multi-turn with an image

We can attach an image to the **last** user message and ask follow-up questions about it. The image only needs to be sent on the turn it is referenced — subsequent turns can be text-only and the conversation history still mentions the image through the assistant's previous descriptions.

In [ ]:
history = []

# Turn 1 — image + question
history.append({"role": "user", "content": "Describe this image in one sentence."})
reply = chat(history, image_base64=img_b64, max_new_tokens=60)
history.append({"role": "assistant", "content": reply})
print("Assistant:", reply)

# Turn 2 — follow-up, text-only
history.append({"role": "user", "content": "What is its main color?"})
reply = chat(history, max_new_tokens=40)
history.append({"role": "assistant", "content": reply})
print("Assistant:", reply)

## 6. Try your own image

Replace the synthetic image with a real one and ask whatever you like.

In [ ]:
# my_img = Image.open("path/to/your/image.jpg")
# my_b64 = encode_image(my_img)
# print(chat(
#     [{"role": "user", "content": "What is happening in this photo?"}],
#     image_base64=my_b64,
#     max_new_tokens=120,
# ))